## Category Revenue ETL Pipeline

A retail company has provided you with two datasets on GitHub :

orders.csv : containing customer order details:
order_id, product_id, quantity, order_date, region

products.csv : containing product information
product_id, category, price

Your task is to perform data cleaning and revenue analysis by building an ETL code that performs the following operations:

### CSV in Github -> Data cleaning(Pyspark) -> Data transformation(Pyspark) -> Output
 

**1. Extract/Read both CSV files to DataFrame**

Read both the datasets into dataframes directly from the GitHub URLs.

**2. Transform the data**
**A. Data Cleaning**
In the orders dataset:
- Fill missing quantity values with 1
- Convert quantity column to integer data type

In the products dataset:
- Fill missing category values with “unknown”

**B. Merge**
Perform an INNER JOIN between orders and products on product_id

**C. Create Derived Column**
revenue = quantity * price

**D. Aggregation**
Compute total revenue for each category using groupby

**3. Output**

Print the final aggregated DataFrame with the following columns: 
category and total_revenue

Sorted by total_revenue in descending order.


In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import IntegerType
from pyspark.sql.functions import col, sum as count_sum
from pyspark import SparkFiles

spark = SparkSession.builder.appName('ETL Pipeline').getOrCreate()

In [0]:
orders_url = 'https://raw.githubusercontent.com/ankitbansal6/namastesql_etl_datasets/refs/heads/main/ta/orders.csv'
products_url = 'https://raw.githubusercontent.com/ankitbansal6/namastesql_etl_datasets/refs/heads/main/ta/products.csv'

In [0]:
#Download file using SparkFiles
spark.sparkContext.addFile(orders_url)
spark.sparkContext.addFile(products_url)

orders_path = SparkFiles.get(orders.csv)
products_path = SparkFiles.get(products.csv)


#Read CSV using Spark
orders_df = spark.read.csv(orders_path, header=True, inferSchema=True)
products_df = spark.read.csv(products_path, header=True, inferSchema=True)

#Data Cleaning
orders_df = orders_df.fillna({'quantity':1})
orders_df = orders_df.withColumn('quantity',col('quantity').cast(IntegerType()))
products_df = products_df.fillna({'category':'unknown'})

#Join
merged_join = orders_df.join(products_df, on='product_id', how='inner')

#Derived Column
merged_df = merged_df.withColumn('revenue',col('quanity') * col('price'))

#Aggregation
agg_df = (
    merged_df.groupBy('category')
    .agg_df(SparkSum('revenue').alias('total_revenue'))
    .orderBy(col('total_revenue').desc())
)

#Output
agg_df.show(truncate=False)